In [0]:
%pip install praw

dbutils.library.restartPython()

In [0]:
# Script 1 — Main Tables (Posts + Comments + Sentences)
# Append version: same logic as overwrite but inserts new rows only via left_anti join

import re
import praw
from pyspark.sql import SparkSession
from pyspark.sql.types import StructType, StructField, StringType, IntegerType
from pyspark.sql.functions import col, lit, to_date, explode, split, trim, length, regexp_replace, udf
from datetime import datetime, timezone

spark = SparkSession.builder.appName("BeautyTalkPH_Main_Append").getOrCreate()

# ── Credentials ───────────────────────────────────────────────────────────────
CLIENT_ID = dbutils.secrets.get(
    scope="reddit-secrets",
    key="client-id"
)

CLIENT_SECRET = dbutils.secrets.get(
    scope="reddit-secrets",
    key="client-secret"
)

USER_AGENT = dbutils.secrets.get(
    scope="reddit-secrets",
    key="user-agent"
)

reddit = praw.Reddit(
    client_id=CLIENT_ID,
    client_secret=CLIENT_SECRET,
    user_agent=USER_AGENT,
)

# ── Config ────────────────────────────────────────────────────────────────────
SUBREDDIT       = "beautytalkph"
DELTA_DATABASE  = "beauty_analytics"
POSTS_TABLE     = f"{DELTA_DATABASE}.posts"
COMMENTS_TABLE  = f"{DELTA_DATABASE}.comments"
POST_SENT_TABLE = f"{DELTA_DATABASE}.post_sentences"
COMM_SENT_TABLE = f"{DELTA_DATABASE}.comment_sentences"
FLAIR_FILTER    = "review"
TOP_LIMIT       = 2000

# ── Brand Lists ───────────────────────────────────────────────────────────────
local_brands = [
    "ever bilena", "blk", "colourette", "filipinta beauty", "lip pinas",
    "clochefame", "cloud", "beach born", "sunnies", "belo",
    "happy skin", "issy", "lucky beauty", "popique",
    "dr. sensitive", "grwm", "teviant", "avon", "vice", "enigma",
    "ellana", "careline", "shawill", "miniso", "kkv",
    "chuchu", "detail", "dazzle me", "strokes", "time phoria",
    "cloudbeauty", "rmt", "imma", "nichido", "absidy", "squad"
]

international_brands = [
    "3ce", "apieu", "armani", "barenbliss", "bbia", "benefit", "canmake",
    "carslan", "catkin", "cacharel", "carolina herrera", "cezanne",
    "charlotte tilbury", "chifure", "clio", "clinique", "colorkey",
    "colourpop", "dasique", "dior", "elf", "espoir", "estee lauder",
    "etude house", "excel", "fenty", "florasis", "flower knows", "focallure",
    "gisou", "glossier", "heimish", "hold live", "holika holika", "hoola",
    "huda beauty", "hince", "into you", "jmcy", "judydoll", "kate", "kiko",
    "kiss me", "kose", "kosas", "kylie", "laura mercier", "laneige",
    "lancome", "loreal", "luxe cosmetics", "mac", "magefy", "maange",
    "majolica majorca", "make up for ever", "mao ge ping", "marc jacobs",
    "mario", "maybelline", "milk", "missha", "monotheme", "moonshot",
    "morphe", "nars", "opera", "otwoo", "patrick ta", "perfect diary",
    "peripera", "pinkflash", "rare beauty", "revlon", "rhode", "romand",
    "sace lady", "sasi", "sea makeup", "shiseido", "sheglam",
    "summer fridays", "the saem", "tony moly", "too faced", "tower 28",
    "vdl", "visee", "ysl", "zenn"
]

brands_list = local_brands + international_brands

# ── Keyword Lists ─────────────────────────────────────────────────────────────
makeup_categories = [
    "foundation", "concealer", "primer", "powder", "blush", "highlighter",
    "bronzer", "contour", "setting spray", "fixing spray", "eyeshadow",
    "eyeliner", "mascara", "eyebrow", "brow gel", "brow pencil", "lash",
    "eye primer", "palette", "lipstick", "lip gloss", "lip liner", "lip balm",
    "lip oil", "lip plumper", "lip tint", "lip lacquer", "lip stain",
    "skin tint", "brush", "sponge", "eyelash curler", "tweezer",
    "makeup remover", "bb cream", "cc cream", "dd cream", "moisturizer",
    "color corrector", "multistick", "sunscreen"
]

positive_keywords = [
    "love", "loved", "great", "amazing", "excellent", "awesome", "perfect",
    "good", "nice", "best", "beautiful", "fantastic", "recommend", "worth it",
    "holy grail", "repurchase", "favorite", "favourite", "impressed",
    "stunning", "flawless", "lightweight", "comfortable", "long lasting",
    "long-lasting", "affordable", "value", "works", "effective", "happy",
    "obsessed", "glowy", "natural", "smooth", "soft", "hydrating",
    "maganda", "magaling", "ganda", "galing", "sulit", "bet", "winner",
    "solid", "grabeh", "grabe", "astig", "okay siya", "ok siya",
    "nagustuhan", "gusto ko", "love na love", "swak", "bagay", "ayos",
    "magaan", "maayos", "worth", "worth it talaga",
]

negative_keywords = [
    "bad", "worst", "terrible", "awful", "hate", "dislike", "disappointed",
    "disappointing", "poor", "waste", "overpriced", "expensive", "cakey",
    "creasy", "crease", "oxidize", "oxidized", "patchy", "streaky",
    "heavy", "greasy", "oily", "dry", "drying", "irritated", "irritating",
    "broke out", "breakout", "pimple", "allergy", "allergic",
    "not worth", "would not recommend", "return", "fake", "expired",
    "hindi maganda", "hindi okay", "hindi ok", "hindi sulit", "mahal",
    "pangit", "basura", "hindi bet", "nakaka-off", "nakakainis",
    "nakaka-irritate", "sumabog", "nag-oxidize", "hindi maayos",
    "peke", "bigo", "nabigo", "hindi worth it", "sayang", "ayaw ko",
    "di maganda", "di okay", "di ok", "di sulit", "di bet",
]

# ── Pre-compile regex patterns ────────────────────────────────────────────────
brands_patterns   = [(b, re.compile(r'\b' + re.escape(b) + r'\b')) for b in brands_list]
category_patterns = [(c, re.compile(r'\b' + re.escape(c) + r'\b')) for c in makeup_categories]
platform_patterns = [(p, re.compile(r'\b' + re.escape(p) + r'\b')) for p in ["tiktok", "shopee", "lazada"]]
positive_patterns = [re.compile(r'\b' + re.escape(k) + r'\b') for k in positive_keywords]
negative_patterns = [re.compile(r'\b' + re.escape(k) + r'\b') for k in negative_keywords]

# ── Helper Functions ──────────────────────────────────────────────────────────
def detect_brands(text):
    if not text: return None
    t = text.lower()
    found = [b for b, pat in brands_patterns if pat.search(t)]
    return ", ".join(found) if found else None

def detect_category(text):
    if not text: return None
    t = text.lower()
    found = [c for c, pat in category_patterns if pat.search(t)]
    return ", ".join(found) if found else None

def detect_platforms(text):
    if not text: return None
    t = text.lower()
    found = [p for p, pat in platform_patterns if pat.search(t)]
    return ", ".join(found) if found else None

def detect_sentiment(text):
    if not text: return "neutral"
    t = text.lower()
    pos = sum(1 for pat in positive_patterns if pat.search(t))
    neg = sum(1 for pat in negative_patterns if pat.search(t))
    if pos > neg:   return "positive"
    elif neg > pos: return "negative"
    else:           return "neutral"

def ts_to_date(utc_ts):
    return datetime.utcfromtimestamp(utc_ts).strftime("%Y-%m-%d")

detect_brands_udf    = udf(detect_brands,    StringType())
detect_category_udf  = udf(detect_category,  StringType())
detect_platforms_udf = udf(detect_platforms, StringType())
detect_sentiment_udf = udf(detect_sentiment, StringType())

# ── Scrape ────────────────────────────────────────────────────────────────────
subreddit     = reddit.subreddit(SUBREDDIT)
posts_rows    = []
comments_rows = []

print("🔍 Scraping all historical posts...")

for submission in subreddit.top(limit=TOP_LIMIT):
    flair = submission.link_flair_text
    if not flair or FLAIR_FILTER not in flair.lower():
        continue

    post_content = submission.title
    if submission.selftext:
        post_content += " " + submission.selftext

    posts_rows.append((
        submission.id,
        post_content,
        submission.url,
        int(submission.score),
        int(submission.num_comments),
        ts_to_date(submission.created_utc),
    ))

    submission.comments.replace_more(limit=0)
    for comment in submission.comments.list():
        comments_rows.append((
            submission.id,
            comment.id,
            comment.body,
            int(comment.score),
            ts_to_date(comment.created_utc),
        ))

print(f"✅ Found {len(posts_rows)} posts, {len(comments_rows)} comments.")

# ── Schemas ───────────────────────────────────────────────────────────────────
posts_schema = StructType([
    StructField("post_id",            StringType(),  False),
    StructField("post_content",       StringType(),  True),
    StructField("post_url",           StringType(),  True),
    StructField("number_of_upvotes",  IntegerType(), True),
    StructField("number_of_comments", IntegerType(), True),
    StructField("date",               StringType(),  True),
])

comments_schema = StructType([
    StructField("post_id",         StringType(),  False),
    StructField("comment_id",      StringType(),  False),
    StructField("comment_body",    StringType(),  True),
    StructField("comment_upvotes", IntegerType(), True),
    StructField("date",            StringType(),  True),
])

# ── Build DataFrames ──────────────────────────────────────────────────────────
posts_df    = spark.createDataFrame(posts_rows,    schema=posts_schema)
comments_df = spark.createDataFrame(comments_rows, schema=comments_schema)

posts_df    = posts_df.withColumn("date",    to_date(col("date"), "yyyy-MM-dd"))
comments_df = comments_df.withColumn("date", to_date(col("date"), "yyyy-MM-dd"))

posts_df = posts_df \
    .withColumn("brands_mentioned",   detect_brands_udf(col("post_content"))) \
    .withColumn("category_mentioned", detect_category_udf(col("post_content"))) \
    .withColumn("platform_mentioned", detect_platforms_udf(col("post_content")))

comments_df = comments_df \
    .withColumn("brands_mentioned",   detect_brands_udf(col("comment_body"))) \
    .withColumn("category_mentioned", detect_category_udf(col("comment_body"))) \
    .withColumn("platform_mentioned", detect_platforms_udf(col("comment_body")))

# ── Append Posts (new post_ids only) ─────────────────────────────────────────
print("📥 Appending new posts...")

existing_post_ids = spark.table(POSTS_TABLE).select("post_id").distinct()
new_posts_df      = posts_df.join(existing_post_ids, on="post_id", how="left_anti")

new_posts_df.write \
    .format("delta") \
    .mode("append") \
    .saveAsTable(POSTS_TABLE)

print(f"✅ Posts appended: {new_posts_df.count()} new rows")

# ── Append Comments (new comment_ids only) ────────────────────────────────────
print("📥 Appending new comments...")

existing_comment_ids = spark.table(COMMENTS_TABLE).select("comment_id").distinct()
new_comments_df      = comments_df.join(existing_comment_ids, on="comment_id", how="left_anti")

new_comments_df.write \
    .format("delta") \
    .mode("append") \
    .saveAsTable(COMMENTS_TABLE)

print(f"✅ Comments appended: {new_comments_df.count()} new rows")

# ── Append Post Sentences (new post_ids only) ─────────────────────────────────
print("🔍 Building post sentences...")

existing_sent_post_ids = spark.table(POST_SENT_TABLE).select("post_id").distinct()
new_posts_for_sent     = posts_df.join(existing_sent_post_ids, on="post_id", how="left_anti")

post_sentences_df = new_posts_for_sent \
    .select("post_id", "post_content") \
    .withColumn("post_content", regexp_replace(col("post_content"), r"\n+", " ")) \
    .withColumn("sentence", explode(split(col("post_content"), r"(?<=[.!?])\s+"))) \
    .withColumn("sentence", trim(col("sentence"))) \
    .filter(length(col("sentence")) > 5) \
    .drop("post_content") \
    .dropDuplicates(["post_id", "sentence"]) \
    .withColumn("brand_found",        detect_brands_udf(col("sentence"))) \
    .withColumn("category_mentioned", detect_category_udf(col("sentence"))) \
    .withColumn("platform_mentioned", detect_platforms_udf(col("sentence"))) \
    .withColumn("sentiment",          detect_sentiment_udf(col("sentence"))) \
    .withColumn("date",               to_date(lit(datetime.now(timezone.utc).strftime("%Y-%m-%d")), "yyyy-MM-dd")) \
    .filter(col("brand_found").isNotNull())

post_sentences_df.write \
    .format("delta") \
    .mode("append") \
    .saveAsTable(POST_SENT_TABLE)

print(f"✅ Post sentences appended: {post_sentences_df.count()} new rows")

# ── Append Comment Sentences (new comment_ids only) ───────────────────────────
print("🔍 Building comment sentences...")

existing_sent_comment_ids = spark.table(COMM_SENT_TABLE).select("comment_id").distinct()
new_comments_for_sent     = comments_df.join(existing_sent_comment_ids, on="comment_id", how="left_anti")

comment_sentences_df = new_comments_for_sent \
    .select("post_id", "comment_id", "comment_body") \
    .withColumn("comment_body", regexp_replace(col("comment_body"), r"\n+", " ")) \
    .withColumn("sentence", explode(split(col("comment_body"), r"(?<=[.!?])\s+"))) \
    .withColumn("sentence", trim(col("sentence"))) \
    .filter(length(col("sentence")) > 5) \
    .drop("comment_body") \
    .dropDuplicates(["comment_id", "sentence"]) \
    .withColumn("brand_found",        detect_brands_udf(col("sentence"))) \
    .withColumn("category_mentioned", detect_category_udf(col("sentence"))) \
    .withColumn("platform_mentioned", detect_platforms_udf(col("sentence"))) \
    .withColumn("sentiment",          detect_sentiment_udf(col("sentence"))) \
    .withColumn("date",               to_date(lit(datetime.now(timezone.utc).strftime("%Y-%m-%d")), "yyyy-MM-dd")) \
    .filter(col("brand_found").isNotNull())

comment_sentences_df.write \
    .format("delta") \
    .mode("append") \
    .saveAsTable(COMM_SENT_TABLE)

print(f"✅ Comment sentences appended: {comment_sentences_df.count()} new rows")

# ── Summary ───────────────────────────────────────────────────────────────────
print("\n🎉 Append run complete!")
print(f"   Posts:             {spark.table(POSTS_TABLE).count()} total rows")
print(f"   Comments:          {spark.table(COMMENTS_TABLE).count()} total rows")
print(f"   Post sentences:    {spark.table(POST_SENT_TABLE).count()} total rows")
print(f"   Comment sentences: {spark.table(COMM_SENT_TABLE).count()} total rows")

In [0]:
# Script 2 — Post Dimension Unpivot Tables
# Append version:
#   - left_anti on post_id → skip posts already in each unpivot table
#   - dropDuplicates on (post_id, brand/category/platform) → no duplicates within the batch

from pyspark.sql import SparkSession
from pyspark.sql.functions import col, split, explode, trim, when

spark = SparkSession.builder.appName("BeautyTalkPH_PostUnpivot_Append").getOrCreate()

# ── Config ────────────────────────────────────────────────────────────────────
DELTA_DATABASE         = "beauty_analytics"
POSTS_TABLE            = f"{DELTA_DATABASE}.posts"
BRANDS_UNPIVOT_TABLE   = f"{DELTA_DATABASE}.post_brands_mentioned"
CATEGORY_UNPIVOT_TABLE = f"{DELTA_DATABASE}.post_categories_mentioned"
PLATFORM_UNPIVOT_TABLE = f"{DELTA_DATABASE}.post_platforms_mentioned"

# ── Brand Lists ───────────────────────────────────────────────────────────────
local_brands = [
    "ever bilena", "blk", "colourette", "filipinta beauty", "lip pinas",
    "clochefame", "cloud", "beach born", "sunnies", "belo",
    "happy skin", "issy", "lucky beauty", "popique",
    "dr. sensitive", "grwm", "teviant", "avon", "vice", "enigma",
    "ellana", "careline", "shawill", "miniso", "kkv",
    "chuchu", "detail", "dazzle me", "strokes", "time phoria",
    "cloudbeauty", "rmt", "imma", "nichido", "absidy", "squad"
]

international_brands = [
    "3ce", "apieu", "armani", "barenbliss", "bbia", "benefit", "canmake",
    "carslan", "catkin", "cacharel", "carolina herrera", "cezanne",
    "charlotte tilbury", "chifure", "clio", "clinique", "colorkey",
    "colourpop", "dasique", "dior", "elf", "espoir", "estee lauder",
    "etude house", "excel", "fenty", "florasis", "flower knows", "focallure",
    "gisou", "glossier", "heimish", "hold live", "holika holika", "hoola",
    "huda beauty", "hince", "into you", "jmcy", "judydoll", "kate", "kiko",
    "kiss me", "kose", "kosas", "kylie", "laura mercier", "laneige",
    "lancome", "loreal", "luxe cosmetics", "mac", "magefy", "maange",
    "majolica majorca", "make up for ever", "mao ge ping", "marc jacobs",
    "mario", "maybelline", "milk", "missha", "monotheme", "moonshot",
    "morphe", "nars", "opera", "otwoo", "patrick ta", "perfect diary",
    "peripera", "pinkflash", "rare beauty", "revlon", "rhode", "romand",
    "sace lady", "sasi", "sea makeup", "shiseido", "sheglam",
    "summer fridays", "the saem", "tony moly", "too faced", "tower 28",
    "vdl", "visee", "ysl", "zenn"
]

# ── Unpivot Brands ────────────────────────────────────────────────────────────
print("🔍 Processing new post brands...")

existing_brand_post_ids = spark.table(BRANDS_UNPIVOT_TABLE).select("post_id").distinct()

new_brands_df = spark.table(POSTS_TABLE) \
    .select("post_id", "brands_mentioned") \
    .join(existing_brand_post_ids, on="post_id", how="left_anti") \
    .filter(col("brands_mentioned").isNotNull()) \
    .withColumn("brand", explode(split(col("brands_mentioned"), ","))) \
    .withColumn("brand", trim(col("brand"))) \
    .withColumn("origin", when(col("brand").isin(local_brands), "local")
                          .when(col("brand").isin(international_brands), "international")
                          .otherwise("unknown")) \
    .select("post_id", "brand", "origin") \
    .dropDuplicates(["post_id", "brand"])

new_brands_df.write \
    .format("delta") \
    .mode("append") \
    .saveAsTable(BRANDS_UNPIVOT_TABLE)

print(f"✅ Post brands appended: {new_brands_df.count()} new rows")

# ── Unpivot Categories ────────────────────────────────────────────────────────
print("🔍 Processing new post categories...")

existing_category_post_ids = spark.table(CATEGORY_UNPIVOT_TABLE).select("post_id").distinct()

new_categories_df = spark.table(POSTS_TABLE) \
    .select("post_id", "category_mentioned") \
    .join(existing_category_post_ids, on="post_id", how="left_anti") \
    .filter(col("category_mentioned").isNotNull()) \
    .withColumn("category", explode(split(col("category_mentioned"), ","))) \
    .withColumn("category", trim(col("category"))) \
    .select("post_id", "category") \
    .dropDuplicates(["post_id", "category"])

new_categories_df.write \
    .format("delta") \
    .mode("append") \
    .saveAsTable(CATEGORY_UNPIVOT_TABLE)

print(f"✅ Post categories appended: {new_categories_df.count()} new rows")

# ── Unpivot Platforms ─────────────────────────────────────────────────────────
print("🔍 Processing new post platforms...")

existing_platform_post_ids = spark.table(PLATFORM_UNPIVOT_TABLE).select("post_id").distinct()

new_platforms_df = spark.table(POSTS_TABLE) \
    .select("post_id", "platform_mentioned") \
    .join(existing_platform_post_ids, on="post_id", how="left_anti") \
    .filter(col("platform_mentioned").isNotNull()) \
    .withColumn("platform", explode(split(col("platform_mentioned"), ","))) \
    .withColumn("platform", trim(col("platform"))) \
    .select("post_id", "platform") \
    .dropDuplicates(["post_id", "platform"])

new_platforms_df.write \
    .format("delta") \
    .mode("append") \
    .saveAsTable(PLATFORM_UNPIVOT_TABLE)

print(f"✅ Post platforms appended: {new_platforms_df.count()} new rows")

# ── Summary ───────────────────────────────────────────────────────────────────
print("\n🎉 Post unpivot append complete!")
print(f"   Brands:     {spark.table(BRANDS_UNPIVOT_TABLE).count()} total rows")
print(f"   Categories: {spark.table(CATEGORY_UNPIVOT_TABLE).count()} total rows")
print(f"   Platforms:  {spark.table(PLATFORM_UNPIVOT_TABLE).count()} total rows")

In [0]:
# Script 3 — Comment Dimension Unpivot Tables
# Append version:
#   - left_anti on comment_id → skip comments already in each unpivot table
#   - dropDuplicates on (comment_id, brand/category/platform) → no duplicates within the batch

from pyspark.sql import SparkSession
from pyspark.sql.functions import col, split, explode, trim

spark = SparkSession.builder.appName("BeautyTalkPH_CommentUnpivot_Append").getOrCreate()

# ── Config ────────────────────────────────────────────────────────────────────
DELTA_DATABASE                 = "beauty_analytics"
COMMENTS_TABLE                 = f"{DELTA_DATABASE}.comments"
COMMENT_BRANDS_UNPIVOT_TABLE   = f"{DELTA_DATABASE}.comment_brands_mentioned"
COMMENT_CATEGORY_UNPIVOT_TABLE = f"{DELTA_DATABASE}.comment_categories_mentioned"
COMMENT_PLATFORM_UNPIVOT_TABLE = f"{DELTA_DATABASE}.comment_platforms_mentioned"

# ── Unpivot Brands ────────────────────────────────────────────────────────────
print("🔍 Processing new comment brands...")

existing_brand_comment_ids = spark.table(COMMENT_BRANDS_UNPIVOT_TABLE).select("comment_id").distinct()

new_comment_brands_df = spark.table(COMMENTS_TABLE) \
    .select("post_id", "comment_id", "brands_mentioned") \
    .join(existing_brand_comment_ids, on="comment_id", how="left_anti") \
    .filter(col("brands_mentioned").isNotNull()) \
    .withColumn("brand", explode(split(col("brands_mentioned"), ","))) \
    .withColumn("brand", trim(col("brand"))) \
    .select("post_id", "comment_id", "brand") \
    .dropDuplicates(["comment_id", "brand"])

new_comment_brands_df.write \
    .format("delta") \
    .mode("append") \
    .saveAsTable(COMMENT_BRANDS_UNPIVOT_TABLE)

print(f"✅ Comment brands appended: {new_comment_brands_df.count()} new rows")

# ── Unpivot Categories ────────────────────────────────────────────────────────
print("🔍 Processing new comment categories...")

existing_category_comment_ids = spark.table(COMMENT_CATEGORY_UNPIVOT_TABLE).select("comment_id").distinct()

new_comment_categories_df = spark.table(COMMENTS_TABLE) \
    .select("post_id", "comment_id", "category_mentioned") \
    .join(existing_category_comment_ids, on="comment_id", how="left_anti") \
    .filter(col("category_mentioned").isNotNull()) \
    .withColumn("category", explode(split(col("category_mentioned"), ","))) \
    .withColumn("category", trim(col("category"))) \
    .select("post_id", "comment_id", "category") \
    .dropDuplicates(["comment_id", "category"])

new_comment_categories_df.write \
    .format("delta") \
    .mode("append") \
    .saveAsTable(COMMENT_CATEGORY_UNPIVOT_TABLE)

print(f"✅ Comment categories appended: {new_comment_categories_df.count()} new rows")

# ── Unpivot Platforms ─────────────────────────────────────────────────────────
print("🔍 Processing new comment platforms...")

existing_platform_comment_ids = spark.table(COMMENT_PLATFORM_UNPIVOT_TABLE).select("comment_id").distinct()

new_comment_platforms_df = spark.table(COMMENTS_TABLE) \
    .select("post_id", "comment_id", "platform_mentioned") \
    .join(existing_platform_comment_ids, on="comment_id", how="left_anti") \
    .filter(col("platform_mentioned").isNotNull()) \
    .withColumn("platform", explode(split(col("platform_mentioned"), ","))) \
    .withColumn("platform", trim(col("platform"))) \
    .select("post_id", "comment_id", "platform") \
    .dropDuplicates(["comment_id", "platform"])

new_comment_platforms_df.write \
    .format("delta") \
    .mode("append") \
    .saveAsTable(COMMENT_PLATFORM_UNPIVOT_TABLE)

print(f"✅ Comment platforms appended: {new_comment_platforms_df.count()} new rows")

# ── Summary ───────────────────────────────────────────────────────────────────
print("\n🎉 Comment unpivot append complete!")
print(f"   Comment Brands:     {spark.table(COMMENT_BRANDS_UNPIVOT_TABLE).count()} total rows")
print(f"   Comment Categories: {spark.table(COMMENT_CATEGORY_UNPIVOT_TABLE).count()} total rows")
print(f"   Comment Platforms:  {spark.table(COMMENT_PLATFORM_UNPIVOT_TABLE).count()} total rows")